In [1]:
import pandas as pd
import vectorbt as vbt
df_market = pd.read_parquet("data_ohlcv.parquet")
df_onchain = pd.read_parquet("data_mev.parquet")
df_onchain.index = df_onchain.index.tz_localize(None)
df_market['return'] = df_market['close'].pct_change()

In [16]:
import pandas as pd
import numpy as np
import vectorbt as vbt

# 1. 构造/加载数据 (假设你的量价数据 df_market 和链上数据 df_onchain 已经按 time 对齐)
# df_market 应包含: ['open', 'high', 'low', 'close', 'volume']
# df_onchain 应包含: 你贴出的那些 sandwich 相关列

def calculate_onchain_factors_normalized(df_onchain, df_market, window=144):
    """
    window=144 假设是 5min 数据下一整天的时间窗口
    """
    factors = pd.DataFrame(index=df_onchain.index)
    
    # 基础因子计算
    raw_intensity = df_onchain['trades_Sandwich'] / (
        df_onchain['trades_Sandwich'] + df_onchain['trades_Sandwiched']
    ).replace(0, np.nan)
    
    raw_vol_share = (df_onchain['volume_Sandwich'] + df_onchain['volume_Sandwiched']) / df_market['volume']

    # --- 归一化处理：Rolling Z-Score ---
    factors['f_sandwich_intensity_z'] = np.log1p(raw_intensity) * (raw_intensity.rolling(window).std()) * 100
    
    factors['f_mev_vol_share_z'] = (raw_vol_share) / (raw_vol_share.rolling(window).std()+0.01)

    raw_eff = df_onchain['volume_Sandwiched'] / df_onchain['volume_Sandwich'].replace(0, np.nan)
    factors['f_attack_eff_z'] = (raw_eff) / (raw_eff.rolling(window).std()+0.01)
    

    return factors.fillna(0)

# 2. IC 测试逻辑
def analyze_factor_ic(factor_df, price_series, forward_window=1):
    """
    计算因子与未来 N 期收益率的 Rank IC
    forward_window: 如果是 5min 数据，12 代表预测未来 1 小时
    """
    # 计算未来收益率 (Forward Returns)
    # vbt 逻辑：shift(-N) 将未来的收益移动到当前行，以便计算 correlation
    forward_returns = price_series.pct_change().shift(-forward_window).abs()
    ic_results = {}
    for col in factor_df.columns:
        # 计算 Rank IC (Spearman 相关系数)
        ic = factor_df[col].corr(forward_returns, method='spearman')
        ic_results[col] = ic
        
    return pd.Series(ic_results)

# --- 实战运行示例 ---

# 假设 price 是你的收盘价序列
price = df_market['close']

# 第一步：生成因子库
my_factors = calculate_onchain_factors_normalized(df_onchain, df_market)

# 第二步：计算各因子的 IC 值
ic_summary = analyze_factor_ic(my_factors, price)

print("因子 Rank IC 测试结果：")
print(ic_summary)



因子 Rank IC 测试结果：
f_sandwich_intensity_z   -0.154179
f_mev_vol_share_z        -0.086128
f_attack_eff_z           -0.098676
dtype: float64


In [17]:
my_factors.describe()

,f_sandwich_intensity_z,f_mev_vol_share_z,f_attack_eff_z
count,731957.000000,731957.000000,731957.000000
mean,0.759397,0.319249,0.542459
std,0.913112,0.836765,0.938161
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.393860,0.007334,0.034958
75%,1.295810,0.242306,0.796136
max,8.978534,12.035910,12.107338


In [18]:
my_factors.to_parquet("factors_mev_new.parquet")